# Lab 5 — продолжение эксперимента
Загрузите lab5-checkpoint.zip. Выполняйте ячейки по порядку. Основной прогон выполняет до 72 запросов (часть уже сохранена), поэтому бесплатный API может потребовать нескольких сессий. Отчёты составляются после получения результатов.

In [ ]:
from google.colab import files, userdata
from pathlib import Path
import zipfile, os, json, sys
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
project = Path("/content/lab5-project")
project.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        target = (project / item.filename).resolve()
        if not target.is_relative_to(project.resolve()):
            raise ValueError("Некорректный путь в архиве")
        if ".git" not in Path(item.filename).parts and "__pycache__" not in Path(item.filename).parts:
            z.extract(item, project)
os.chdir(project)
sys.path.insert(0, str(project))
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
print("Архив загружен, ключ подключён.")


Saving lab5-checkpoint.zip to lab5-checkpoint.zip
Архив загружен, ключ подключён.


In [ ]:
import random, time, shutil
from ml.models.api_client import ExperimentClient
from ml.prompts.lab5 import classify_prompt, summarize_prompt

dataset = json.loads(Path("evals/dataset.json").read_text())
client = ExperimentClient()
tasks = []
for game in dataset:
    for variant in ["lab3", "lab4", "final"]:
        tasks.append((game, "classify", variant))
    for variant in ["lab4", "final", "without_grounding"]:
        tasks.append((game, "summarize", variant))
random.Random(42).shuffle(tasks)
records = []
try:
    for i, (game, task, variant) in enumerate(tasks, 1):
        prompt = (classify_prompt(game["game_id"], game["metrics_4w"], variant)
                  if task == "classify" else
                  summarize_prompt(game["game_id"], game["store_description"], variant))
        print(f"{i}/{len(tasks)} {game['game_id']} {task} {variant}", flush=True)
        try:
            record = client.call(prompt, task, game["game_id"], variant, repeat=0)
        except RuntimeError as exc:
            print("Остановка:", exc)
            break
        records.append(record)
        Path("docs/experiments/main-records.json").write_text(
            json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Сохранено; finish_reason:", record["finish_reason"],
              "; с диска:", record["resumed_from_disk"], flush=True)
        if not record["resumed_from_disk"]:
            time.sleep(60)
finally:
    archive = shutil.make_archive("/content/lab5-results", "zip", root_dir=project)
    print("Архив результатов готов. Скачайте следующей ячейкой.")


1/72 noise_01 classify lab4
HTTP 429: повтор через 15 с
Сохранено; finish_reason: stop ; с диска: False
2/72 synth_004 classify lab3
Сохранено; finish_reason: length ; с диска: True
3/72 decline_01 classify lab3
Сохранено; finish_reason: stop ; с диска: False
4/72 flat_01 summarize final
HTTP 429: повтор через 15 с
HTTP 429: повтор через 30 с
HTTP 429: повтор через 45 с
Остановка: HTTP 429; запрос не завершён. Успешные результаты сохранены.
Архив результатов готов. Скачайте следующей ячейкой.


In [ ]:
# Можно выполнить отдельно после остановки прогона.
import shutil
from google.colab import files
shutil.make_archive("/content/lab5-results", "zip", root_dir=project)
files.download("/content/lab5-results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import subprocess

result = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
)

if result.returncode == 0:
    print("Видеокарта:", result.stdout.strip())
else:
    print("GPU не подключён.")
    print(result.stderr)

Видеокарта: Tesla T4, 15360 MiB


In [2]:
%pip -q install "transformers==4.48.3" "accelerate==1.3.0"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

assert torch.cuda.is_available(), "GPU не подключён"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model.eval()

print("Модель загружена:", MODEL_ID)
print("Устройство:", next(model.parameters()).device)
print(
    "Занято памяти GPU:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "ГБ",
)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Модель загружена: Qwen/Qwen2.5-3B-Instruct
Устройство: cuda:0
Занято памяти GPU: 5.85 ГБ


In [3]:
import json
import time
import torch

def generate_local(prompt, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    torch.cuda.synchronize()
    started = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    generated = outputs[0, inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(
        generated, skip_special_tokens=True
    )

    return {
        "response": response,
        "latency_s": round(elapsed, 3),
        "prompt_tokens": inputs["input_ids"].shape[1],
        "completion_tokens": len(generated),
        "finish_reason": (
            "stop"
            if len(generated) and
               generated[-1].item() == tokenizer.eos_token_id
            else "length"
        ),
    }

test_prompt = """Ты — продуктовый аналитик.
Определи категорию по четырём недельным значениям выручки.

green: каждое значение строго больше предыдущего,
а общий рост не менее 15%.
red: каждое значение строго меньше предыдущего,
а общее падение не менее 15%.
yellow: все остальные случаи, включая разовый всплеск.

Метрики: [100, 95, 90, 200].
Верни только JSON:
{"game_id": "so_003", "category": "выбранная категория"}
"""

test_result = generate_local(test_prompt)
print(json.dumps(test_result, ensure_ascii=False, indent=2))

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


{
  "response": "{\"game_id\": \"so_003\", \"category\": \"red\"}",
  "latency_s": 8.019,
  "prompt_tokens": 180,
  "completion_tokens": 18,
  "finish_reason": "stop"
}


In [4]:
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
import zipfile
import os
import sys
import json
import random
import time
import hashlib
import shutil
import platform
import transformers
import torch

# 1. Загрузка ранее сохранённого проекта
print("Выбери файл lab5-results.zip")
uploaded = files.upload()
archive_name = next(
    name for name in uploaded if name.endswith(".zip")
)

project = Path("/content/lab5-qwen")
project.mkdir(exist_ok=True)

with zipfile.ZipFile(archive_name) as archive:
    for item in archive.infolist():
        target = (project / item.filename).resolve()
        if not target.is_relative_to(project.resolve()):
            raise ValueError("Некорректный путь в архиве")
        if not {".git", "__pycache__"}.intersection(
            Path(item.filename).parts
        ):
            archive.extract(item, project)

os.chdir(project)
sys.path.insert(0, str(project))

from ml.prompts.lab5 import classify_prompt, summarize_prompt

dataset = json.loads(
    Path("evals/dataset.json").read_text(encoding="utf-8")
)

output_dir = Path("docs/experiments/qwen")
output_dir.mkdir(parents=True, exist_ok=True)

# 2. Одинаковые параметры для всех вариантов
generation_config = {
    "max_new_tokens": 256,
    "do_sample": False,
    "temperature": None,
    "top_p": None,
    "top_k": None,
    "pad_token_id": tokenizer.eos_token_id,
}

environment = {
    "model": MODEL_ID,
    "model_commit": getattr(model.config, "_commit_hash", None),
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "dtype": str(next(model.parameters()).dtype),
    "generation": generation_config,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "dataset_sha256": hashlib.sha256(
        Path("evals/dataset.json").read_bytes()
    ).hexdigest(),
}

(output_dir / "environment.json").write_text(
    json.dumps(environment, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Сохраняем результат первой проверки отдельно.
(output_dir / "smoke-test.json").write_text(
    json.dumps(
        {"prompt": test_prompt, **test_result},
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

def qwen_generate(prompt):
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    torch.cuda.synchronize()
    started = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs, **generation_config
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    generated = outputs[0, inputs["input_ids"].shape[1]:]

    return {
        "response": tokenizer.decode(
            generated, skip_special_tokens=True
        ),
        "latency_s": round(elapsed, 4),
        "prompt_tokens": int(inputs["input_ids"].shape[1]),
        "completion_tokens": int(len(generated)),
        "finish_reason": (
            "stop"
            if len(generated)
            and generated[-1].item() == tokenizer.eos_token_id
            else "length"
        ),
    }

# Прогрев GPU не включаем в метрики эксперимента.
print("Прогрев модели…", flush=True)
qwen_generate("Ответь одним словом: привет.")

tasks = []
for game in dataset:
    for variant in ["lab3", "lab4", "final"]:
        tasks.append((game, "classify", variant))
    for variant in ["lab4", "final", "without_grounding"]:
        tasks.append((game, "summarize", variant))

random.Random(42).shuffle(tasks)
records = []

try:
    for index, (game, task, variant) in enumerate(tasks, 1):
        prompt = (
            classify_prompt(
                game["game_id"], game["metrics_4w"], variant
            )
            if task == "classify"
            else summarize_prompt(
                game["game_id"],
                game["store_description"],
                variant,
            )
        )

        identity = {
            "model": environment["model"],
            "model_commit": environment["model_commit"],
            "generation": generation_config,
            "prompt": prompt,
            "task": task,
            "game_id": game["game_id"],
            "variant": variant,
        }
        request_id = hashlib.sha256(
            json.dumps(
                identity, ensure_ascii=False, sort_keys=True
            ).encode("utf-8")
        ).hexdigest()

        result_path = output_dir / f"{request_id}.json"

        if result_path.exists():
            record = json.loads(
                result_path.read_text(encoding="utf-8")
            )
            source = "с диска"
        else:
            result = qwen_generate(prompt)
            record = {
                **identity,
                **result,
                "request_id": request_id,
                "api_cost_usd": 0,
                "timestamp":
                    datetime.now(timezone.utc).isoformat(),
            }
            result_path.write_text(
                json.dumps(
                    record, ensure_ascii=False, indent=2
                ),
                encoding="utf-8",
            )
            source = "новый"

        records.append(record)

        (output_dir / "records.json").write_text(
            json.dumps(records, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

        print(
            f"[{index}/{len(tasks)}] "
            f"{game['game_id']} / {task} / {variant}: "
            f"{record['finish_reason']}, "
            f"{record['latency_s']:.2f} с ({source})",
            flush=True,
        )

finally:
    result_archive = shutil.make_archive(
        "/content/lab5-qwen-results",
        "zip",
        root_dir=project,
    )
    print(f"\nСохранено результатов: {len(records)}/{len(tasks)}")
    files.download(result_archive)

Выбери файл lab5-results.zip


Saving lab5-results.zip to lab5-results.zip
Прогрев модели…
[1/72] noise_01 / classify / lab4: stop, 0.94 с (новый)
[2/72] synth_004 / classify / lab3: stop, 1.18 с (новый)
[3/72] decline_01 / classify / lab3: stop, 1.07 с (новый)
[4/72] flat_01 / summarize / final: stop, 1.77 с (новый)
[5/72] tt_001 / classify / lab4: stop, 0.76 с (новый)
[6/72] synth_004 / classify / final: stop, 1.32 с (новый)
[7/72] so_003 / summarize / lab4: stop, 3.26 с (новый)
[8/72] decline_01 / summarize / lab4: stop, 2.21 с (новый)
[9/72] noise_01 / summarize / final: stop, 2.27 с (новый)
[10/72] nr_002 / summarize / final: stop, 2.28 с (новый)
[11/72] decline_02 / classify / lab3: length, 10.89 с (новый)
[12/72] flat_01 / classify / lab3: stop, 0.85 с (новый)
[13/72] flat_01 / classify / lab4: stop, 0.72 с (новый)
[14/72] noise_01 / classify / lab3: stop, 0.86 с (новый)
[15/72] so_003 / summarize / final: stop, 1.88 с (новый)
[16/72] growth_01 / summarize / lab4: stop, 3.31 с (новый)
[17/72] synth_005 / summ

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import json
from pathlib import Path

def classify_trend(values):
    if len(values) != 4:
        raise ValueError("Нужны четыре недельных значения")

    first, last = values[0], values[-1]
    if first <= 0:
        raise ValueError("Начальное значение должно быть положительным")

    increasing = all(b > a for a, b in zip(values, values[1:]))
    decreasing = all(b < a for a, b in zip(values, values[1:]))

    if increasing and last >= first * 1.15:
        return "green"
    if decreasing and last <= first * 0.85:
        return "red"
    return "yellow"

dataset = json.loads(
    Path("evals/dataset.json").read_text(encoding="utf-8")
)

results = [
    {
        "game_id": game["game_id"],
        "expected": game["expected_category"],
        "predicted": classify_trend(game["metrics_4w"]),
    }
    for game in dataset
]

correct = sum(
    row["expected"] == row["predicted"] for row in results
)

print(f"Правила Python: {correct}/{len(results)}")
print("Это проверка реализации заданных правил,")
print("а не доказательство качества на реальных рыночных данных.")

Правила Python: 12/12
Это проверка реализации заданных правил,
а не доказательство качества на реальных рыночных данных.


In [6]:
from pathlib import Path
import subprocess
import sys

repository = (
    "https://github.com/vstu-sii/"
    "nedelnyy-daydzhest-mobilnogo-geyminga-wthe.git"
)
branch = "lab5-ai-engineer-testing"
check_dir = Path("/content/lab5-github-check")

if not check_dir.exists():
    subprocess.run(
        [
            "git", "clone",
            "--branch", branch,
            "--single-branch",
            repository,
            str(check_dir),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(check_dir), "pull", "--ff-only"],
        check=True,
    )

print("\n=== Тесты классификатора, валидатора и пайплайна ===")
tests = subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", "evals",
        "-p", "test_*.py",
        "-v",
    ],
    cwd=check_dir,
    text=True,
    capture_output=True,
)
print(tests.stdout)
print(tests.stderr)

print("\n=== Пересчёт 72 ответов Qwen ===")
metrics = subprocess.run(
    [sys.executable, "evals/summarize_results.py"],
    cwd=check_dir,
    text=True,
    capture_output=True,
)
print(metrics.stdout)
print(metrics.stderr)

print(
    "\nИтог:",
    "ПРОВЕРКИ ПРОЙДЕНЫ"
    if tests.returncode == 0 and metrics.returncode == 0
    else "ЕСТЬ ОШИБКИ — пришли вывод ячейки",
)


=== Тесты классификатора, валидатора и пайплайна ===

test_exact_decline_threshold (test_classification.ClassificationTests.test_exact_decline_threshold) ... ok
test_exact_growth_threshold (test_classification.ClassificationTests.test_exact_growth_threshold) ... ok
test_fixed_dataset (test_classification.ClassificationTests.test_fixed_dataset) ... ok
test_invalid_inputs (test_classification.ClassificationTests.test_invalid_inputs) ... ok
test_plateau_is_yellow (test_classification.ClassificationTests.test_plateau_is_yellow) ... ok
test_builds_draft_card (test_pipeline.PipelineTests.test_builds_draft_card) ... ok
test_empty_input_is_rejected (test_pipeline.PipelineTests.test_empty_input_is_rejected) ... ok
test_invalid_metrics_do_not_call_generator (test_pipeline.PipelineTests.test_invalid_metrics_do_not_call_generator) ... ok
test_model_failure_is_not_hidden (test_pipeline.PipelineTests.test_model_failure_is_not_hidden) ... ok
test_truncated_output_blocks_card (test_pipeline.PipelineT

In [7]:
from google.colab import files

files.download(
    "/content/lab5-github-check/"
    "docs/experiments/qwen/summary.json"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import json
import os
import subprocess
import sys

check_dir = "/content/lab5-github-check"

subprocess.run(
    ["git", "-C", check_dir, "pull", "--ff-only"],
    check=True,
)
os.chdir(check_dir)
sys.path.insert(0, check_dir)

# Используем модули именно из проверяемого репозитория.
for name in list(sys.modules):
    if name == "ml" or name.startswith("ml."):
        del sys.modules[name]

from ml.pipeline import build_game_card
from ml.qwen_generator import QwenSummaryGenerator

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "Модель не загружена. Сообщи об этом — "
        "я дам ячейку повторной загрузки."
    )

generator = QwenSummaryGenerator(model, tokenizer)

card = build_game_card(
    game_id="so_003",
    store_description=(
        "Farm-симулятор. Выращивайте сад, собирайте урожай."
    ),
    metrics_4w=[100, 95, 90, 200],
    generate_summary=generator,
)

assert card["category"] == "yellow"
assert card["publication_status"] == "draft"
assert card["review_required"] is True

print(json.dumps(card, ensure_ascii=False, indent=2))

from pathlib import Path

result_path = Path(
    "docs/experiments/qwen/pipeline-smoke-test.json"
)
result_path.write_text(
    json.dumps(card, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nПроверка финального пайплайна пройдена.")

CalledProcessError: Command '['git', '-C', '/content/lab5-github-check', 'pull', '--ff-only']' returned non-zero exit status 1.

In [10]:
import subprocess
from pathlib import Path

check_dir = Path("/content/lab5-github-check")

print("Папка существует:", check_dir.exists())

if check_dir.exists():
    for command in [
        ["git", "-C", str(check_dir), "status", "--short"],
        ["git", "-C", str(check_dir), "pull", "--ff-only"],
    ]:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
        )
        print("\nКоманда:", " ".join(command))
        print("Код завершения:", result.returncode)
        print(result.stdout)
        print(result.stderr)

Папка существует: True

Команда: git -C /content/lab5-github-check status --short
Код завершения: 0
?? docs/experiments/qwen/summary.json
?? evals/__pycache__/
?? ml/__pycache__/



Команда: git -C /content/lab5-github-check pull --ff-only
Код завершения: 1
Updating 80451f9..415a189

error: The following untracked working tree files would be overwritten by merge:
	docs/experiments/qwen/summary.json
Please move or remove them before you merge.
Aborting



In [11]:
from pathlib import Path
import shutil
import subprocess

project = Path("/content/lab5-github-check")
local_file = project / "docs/experiments/qwen/summary.json"
backup = Path("/content/lab5-summary-local-backup.json")

if local_file.exists():
    shutil.copy2(local_file, backup)
    local_file.unlink()
    print("Локальные метрики сохранены:", backup)

subprocess.run(
    ["git", "-C", str(project), "pull", "--ff-only"],
    check=True,
)

print("Репозиторий обновлён.")

Локальные метрики сохранены: /content/lab5-summary-local-backup.json
Репозиторий обновлён.


In [12]:
from pathlib import Path
import shutil
import subprocess
import sys
import os
import json

project = Path("/content/lab5-github-check")
local_file = project / "docs/experiments/qwen/summary.json"
backup = Path("/content/lab5-summary-local-backup.json")

# Сохраняем локальные метрики перед обновлением.
if local_file.exists():
    shutil.copy2(local_file, backup)
    local_file.unlink()
    print("Резервная копия метрик сохранена.")

subprocess.run(
    ["git", "-C", str(project), "pull", "--ff-only"],
    check=True,
)

os.chdir(project)
sys.path.insert(0, str(project))

# Перезагружаем модули из обновлённого репозитория.
for name in list(sys.modules):
    if name == "ml" or name.startswith("ml."):
        del sys.modules[name]

from ml.pipeline import build_game_card
from ml.qwen_generator import QwenSummaryGenerator

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "Модель Qwen сейчас не загружена. Пришли это сообщение в чат."
    )

generator = QwenSummaryGenerator(model, tokenizer)

card = build_game_card(
    game_id="so_003",
    store_description=(
        "Farm-симулятор. Выращивайте сад, собирайте урожай."
    ),
    metrics_4w=[100, 95, 90, 200],
    generate_summary=generator,
)

assert card["category"] == "yellow"
assert card["publication_status"] == "draft"
assert card["review_required"] is True

result_path = (
    project / "docs/experiments/qwen/pipeline-smoke-test.json"
)
result_path.write_text(
    json.dumps(card, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(card, ensure_ascii=False, indent=2))
print("\nПроверка финального пайплайна пройдена.")

Резервная копия метрик сохранена.
{
  "game_id": "so_003",
  "narrative": "Это симулятор фермы, где игроки выращивают сад и собирают урожай.",
  "category": "yellow",
  "classification_method": "rules-v1",
  "review_required": true,
  "publication_status": "draft",
  "generation_metadata": {
    "latency_s": 2.4674,
    "prompt_tokens": 244,
    "completion_tokens": 45,
    "model": "Qwen/Qwen2.5-3B-Instruct"
  }
}

Проверка финального пайплайна пройдена.
